##  XGBoost + SVM Classifiers

### 1. Setup & Load Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score, confusion_matrix, precision_recall_curve

import xgboost as xgb
from xgboost import XGBClassifier
import optuna

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Load feature-engineered data from the PCA + GMM notebook
X_train_enhanced = pd.read_parquet("../data/processed/X_train_enhanced.parquet")
X_val_enhanced = pd.read_parquet("../data/processed/X_val_enhanced.parquet")
X_test_enhanced = pd.read_parquet("../data/processed/X_test_enhanced.parquet")
y_train = pd.read_parquet("../data/processed/y_train_resampled.parquet").squeeze()
y_val = pd.read_parquet("../data/processed/y_val.parquet").squeeze()
y_test = pd.read_parquet("../data/processed/y_test.parquet").squeeze()

print(f"Enhanced training shape: {X_train_enhanced.shape}")
print(f"Enhanced validation shape: {X_val_enhanced.shape}")
print(f"Enhanced test shape: {X_test_enhanced.shape}")

Enhanced training shape: (797828, 62)
Enhanced validation shape: (59054, 62)
Enhanced test shape: (118108, 62)


### 2. Feature-Engineering Handoff
Notebook 03 already applies train-fitted encoding, imputation, scaling, PCA, and GMM. This notebook starts directly from the saved enhanced datasets.

In [2]:
# Verify the feature-engineering handoff without refitting any transformer
assert X_train_enhanced.isna().sum().sum() == 0, "Enhanced training set still has NaNs!"
assert X_val_enhanced.isna().sum().sum() == 0, "Enhanced validation set still has NaNs!"
assert X_test_enhanced.isna().sum().sum() == 0, "Enhanced test set still has NaNs!"
assert list(X_train_enhanced.columns) == list(X_val_enhanced.columns) == list(X_test_enhanced.columns), "Training, validation, and test features do not match!"
print(" Enhanced datasets are aligned and ready for modeling.")

 Enhanced datasets are aligned and ready for modeling.


### 3. Identify PCA Features for SVM
The PCA features are already present in the saved datasets, so we identify them by name without rebuilding the pipeline.

In [3]:
pca_cols = [col for col in X_train_enhanced.columns if col.startswith('PC')]
n_pca_components = len(pca_cols)
print(f"Using {n_pca_components} principal components from notebook 03.")

Using 17 principal components from notebook 03.


### 4. Modeling Datasets Ready
No transformation is performed here; the enhanced datasets are used directly by the classifiers below.

In [4]:
print(" Enhanced datasets loaded from notebook 03; beginning model training.")


 Enhanced datasets loaded from notebook 03; beginning model training.


### 5. Efficient XGBoost Tuning with Optuna
Hyperparameters are selected on a fixed stratified training sample with an internal validation split. The untouched test set remains reserved for final evaluation.

In [5]:
# Tune on a representative subset to keep the notebook practical on a laptop
TUNING_SAMPLE_SIZE = min(120_000, len(X_train_enhanced))
X_tuning, _, y_tuning, _ = train_test_split(
    X_train_enhanced, y_train, train_size=TUNING_SAMPLE_SIZE,
    stratify=y_train, random_state=42
)
X_tune_train, X_tune_valid, y_tune_train, y_tune_valid = train_test_split(
    X_tuning, y_tuning, test_size=0.20, stratify=y_tuning, random_state=42
)

print(f"Tuning sample: {X_tuning.shape[0]:,} rows ({X_tune_train.shape[0]:,} train / {X_tune_valid.shape[0]:,} validation)")

def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 0.5, 2.0),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'eval_metric': 'logloss',
        'use_label_encoder': False,
        'random_state': 42,
        'enable_categorical': False,
        'tree_method': 'hist',
        'n_jobs': -1
    }
    model = XGBClassifier(**params)
    model.fit(X_tune_train, y_tune_train,
              eval_set=[(X_tune_valid, y_tune_valid)],
              early_stopping_rounds=20, verbose=False)
    preds = model.predict(X_tune_valid)
    return recall_score(y_tune_valid, preds)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=15)

print(f"Best trial: {study.best_trial.params}")
print(f"Best Recall: {study.best_trial.value:.4f}")

[I 2026-08-03 21:17:12,628] A new study created in memory with name: no-name-aeef242b-496f-4766-862c-2e89ac2d7235


Tuning sample: 120,000 rows (96,000 train / 24,000 validation)


[I 2026-08-03 21:17:22,471] Trial 0 finished with value: 0.9698333333333333 and parameters: {'max_depth': 7, 'learning_rate': 0.12716366929280856, 'n_estimators': 134, 'scale_pos_weight': 1.312053101003933, 'subsample': 0.9909897410529115, 'colsample_bytree': 0.8203222626931195}. Best is trial 0 with value: 0.9698333333333333.


[I 2026-08-03 21:17:42,931] Trial 1 finished with value: 0.9153333333333333 and parameters: {'max_depth': 9, 'learning_rate': 0.020311143557760855, 'n_estimators': 171, 'scale_pos_weight': 0.5181671341910818, 'subsample': 0.628919628758542, 'colsample_bytree': 0.7792113828295163}. Best is trial 0 with value: 0.9698333333333333.


[I 2026-08-03 21:18:04,010] Trial 2 finished with value: 0.9346666666666666 and parameters: {'max_depth': 6, 'learning_rate': 0.010762284688945582, 'n_estimators': 287, 'scale_pos_weight': 1.4642603478029108, 'subsample': 0.7262576398495082, 'colsample_bytree': 0.6531416087258027}. Best is trial 0 with value: 0.9698333333333333.


...

[I 2026-08-03 21:20:51,499] Trial 11 finished with value: 0.9785833333333334 and parameters: {'max_depth': 10, 'learning_rate': 0.035863318486346864, 'n_estimators': 236, 'scale_pos_weight': 1.9459232689096704, 'subsample': 0.8587371794887306, 'colsample_bytree': 0.7218432881889183}. Best is trial 11 with value: 0.9785833333333334.


[I 2026-08-03 21:21:14,045] Trial 12 finished with value: 0.9785833333333334 and parameters: {'max_depth': 10, 'learning_rate': 0.037290272914349984, 'n_estimators': 229, 'scale_pos_weight': 1.96527541930919, 'subsample': 0.8497028509908605, 'colsample_bytree': 0.7070250092784653}. Best is trial 11 with value: 0.9785833333333334.


[I 2026-08-03 21:21:34,588] Trial 13 finished with value: 0.9799166666666667 and parameters: {'max_depth': 9, 'learning_rate': 0.06060434631490276, 'n_estimators': 252, 'scale_pos_weight': 1.9435722442194867, 'subsample': 0.8553837172539702, 'colsample_bytree': 0.6833288177113603}. Best is trial 13 with value: 0.9799166666666667.


[I 2026-08-03 21:21:53,533] Trial 14 finished with value: 0.9766666666666667 and parameters: {'max_depth': 8, 'learning_rate': 0.06251714446821127, 'n_estimators': 253, 'scale_pos_weight': 1.7014785918626192, 'subsample': 0.8741487618110788, 'colsample_bytree': 0.6198264393597279}. Best is trial 13 with value: 0.9799166666666667.


Best trial: {'max_depth': 9, 'learning_rate': 0.06060434631490276, 'n_estimators': 252, 'scale_pos_weight': 1.9435722442194867, 'subsample': 0.8553837172539702, 'colsample_bytree': 0.6833288177113603}
Best Recall: 0.9799


### 6. Train Final XGBoost & Evaluate

In [6]:
best_params = study.best_trial.params

# Use a training-only validation split to identify the appropriate boosting round
X_fit, X_valid, y_fit, y_valid = train_test_split(
    X_train_enhanced, y_train, test_size=0.10, stratify=y_train, random_state=42
)
xgb_with_early_stopping = XGBClassifier(
    **best_params, eval_metric='logloss', use_label_encoder=False, random_state=42,
    enable_categorical=False, tree_method='hist', n_jobs=-1
)
xgb_with_early_stopping.fit(
    X_fit, y_fit, eval_set=[(X_valid, y_valid)],
    early_stopping_rounds=20, verbose=False
)

best_iteration = getattr(xgb_with_early_stopping, 'best_iteration', None)
best_n_estimators = best_iteration + 1 if best_iteration is not None else best_params['n_estimators']

# Refit the selected model on all available training data before validation threshold selection
final_params = {**best_params, 'n_estimators': best_n_estimators}
final_xgb = XGBClassifier(
    **final_params, eval_metric='logloss',
    use_label_encoder=False, random_state=42, enable_categorical=False,
    tree_method='hist', n_jobs=-1
)
final_xgb.fit(X_train_enhanced, y_train, verbose=False)

# Select the highest-precision threshold that meets the recall target on untouched validation data
TARGET_RECALL = 0.90
xgb_val_proba = final_xgb.predict_proba(X_val_enhanced)[:, 1]
precision_curve, recall_curve, thresholds = precision_recall_curve(y_val, xgb_val_proba)
eligible = np.flatnonzero(recall_curve[:-1] >= TARGET_RECALL)

if len(eligible) == 0:
    raise ValueError(f"No validation threshold achieved the target recall of {TARGET_RECALL:.0%}.")

best_threshold_idx = eligible[np.argmax(precision_curve[:-1][eligible])]
operating_threshold = thresholds[best_threshold_idx]
validation_preds = (xgb_val_proba >= operating_threshold).astype(int)
validation_recall = recall_score(y_val, validation_preds)
validation_precision = precision_score(y_val, validation_preds, zero_division=0)

xgb_proba = final_xgb.predict_proba(X_test_enhanced)[:, 1]
xgb_preds = (xgb_proba >= operating_threshold).astype(int)

xgb_recall = recall_score(y_test, xgb_preds)
xgb_precision = precision_score(y_test, xgb_preds)
xgb_f1 = f1_score(y_test, xgb_preds)
xgb_auc = roc_auc_score(y_test, xgb_proba)

print(f"Final XGBoost trees: {best_n_estimators}")
print(f"Validation threshold: {operating_threshold:.4f} -> Recall: {validation_recall:.4f}, Precision: {validation_precision:.4f}")
print(f"XGBoost -> Recall: {xgb_recall:.4f}, Precision: {xgb_precision:.4f}, F1: {xgb_f1:.4f}, AUC: {xgb_auc:.4f}")

Final XGBoost trees: 252
Validation threshold: 0.0536 -> Recall: 0.9003, Precision: 0.1088
XGBoost -> Recall: 0.9117, Precision: 0.1109, F1: 0.1977, AUC: 0.9326


### 7. SVM Comparison (LinearSVC on PCA Features)
- We use LinearSVC because RBF SVM is computationally infeasible on 900k samples.
- Since we have 17 orthogonal PCA components, a linear boundary is highly appropriate.
- Hyperparameters are selected on the same stratified tuning sample, then the final model is fit on all training rows.


In [7]:
from sklearn.svm import LinearSVC
from sklearn.model_selection import RandomizedSearchCV

# Use the 17 PCA components
pca_cols = [f"PC{i}" for i in range(1, n_pca_components + 1)]
X_train_svm = X_train_enhanced[pca_cols]
X_test_svm = X_test_enhanced[pca_cols]

# LinearSVC parameters (tuned for speed and accuracy)
param_dist = {
    'C': [0.01, 0.1, 1, 10, 100],  # Regularization strength
    'max_iter': [1000, 2000]  # Ensure convergence
}

# Initialize LinearSVC (dual=False handles n_samples > n_features efficiently)
svm_base = LinearSVC(random_state=42, dual=False)

random_search = RandomizedSearchCV(
    svm_base, 
    param_dist, 
    n_iter=5,            # Small search on the representative tuning sample
    cv=3,                # 3-fold cross-validation
    scoring='recall',    # Optimize for recall
    n_jobs=-1,           # Use all CPU cores
    random_state=42, 
    verbose=1
)

print("Tuning LinearSVC on the representative training sample.")
random_search.fit(X_tuning[pca_cols], y_tuning)

best_svm = LinearSVC(random_state=42, dual=False, **random_search.best_params_)
best_svm.fit(X_train_svm, y_train)
print(f"Best SVM params: {random_search.best_params_}")

# Evaluate
svm_preds = best_svm.predict(X_test_svm)
svm_decision = best_svm.decision_function(X_test_svm)

svm_recall = recall_score(y_test, svm_preds)
svm_precision = precision_score(y_test, svm_preds)
svm_f1 = f1_score(y_test, svm_preds)
svm_auc = roc_auc_score(y_test, svm_decision)

print(f"LinearSVC -> Recall: {svm_recall:.4f}, Precision: {svm_precision:.4f}, F1: {svm_f1:.4f}, AUC: {svm_auc:.4f}")

Tuning LinearSVC on the representative training sample.


Fitting 3 folds for each of 5 candidates, totalling 15 fits


Best SVM params: {'max_iter': 2000, 'C': 0.01}


LinearSVC -> Recall: 0.7815, Precision: 0.0603, F1: 0.1119, AUC: 0.7507


### 8. Results Comparison Table

In [8]:
results = pd.DataFrame({
    'Model': ['XGBoost', 'Linear SVM'],
    'Recall': [xgb_recall, svm_recall],
    'Precision': [xgb_precision, svm_precision],
    'F1': [xgb_f1, svm_f1],
    'AUC-ROC': [xgb_auc, svm_auc]
})
print(results)

        Model    Recall  Precision        F1   AUC-ROC
0     XGBoost  0.911686   0.110869  0.197697  0.932586
1  Linear SVM  0.781515   0.060279  0.111925  0.750744


### 9. Save Models

In [9]:
os.makedirs("../models", exist_ok=True)
joblib.dump(final_xgb, "../models/xgb_model.pkl")
joblib.dump(best_svm, "../models/svm_model.pkl")

print(" Models saved to ../models/")

 Models saved to ../models/


### 10. Operating Threshold Summary
The classification threshold is selected on the untouched validation set to target at least 90% recall, then applied once to the untouched test set.

In [10]:
print(f"Selected operating threshold: {operating_threshold:.4f}")
print(f"Validation recall: {validation_recall:.4f} (target: {TARGET_RECALL:.2f})")
print("The threshold was fixed before final test evaluation; no test labels were used for threshold selection.")

Selected operating threshold: 0.0536
Validation recall: 0.9003 (target: 0.90)
The threshold was fixed before final test evaluation; no test labels were used for threshold selection.


In [11]:
print(" done for fun.")

 done for fun.
